# 📡 LangGraph Stream Writer Tutorial
Learn how to send real-time updates from tools while they run.

In [ ]:
# Cell 1 — Install & Imports
%pip install langchain-openai python-dotenv langgraph -q

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

In [ ]:
# Cell 2 — LLM Setup
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

## 📡 What is Stream Writer?

Normally when a tool runs, the user sees **NOTHING** until it finishes.

**Stream Writer** lets your tool send real-time updates  
WHILE it is still running — like a progress bar.

**How it works:**
1. Get the writer  →  `writer = runtime.stream_writer`
2. Call it anytime →  `writer("your update message")`
3. Return result   →  `return "final result"`

**To receive the updates:**  
Use `agent.stream(..., stream_mode="custom")`  
instead of `agent.invoke(...)`

In [ ]:
# Cell 4 — Basic Stream Writer Tool (from docs)
# Exact example from the docs

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer             # ← get the writer

    writer(f"Looking up data for city: {city}")   # ← send update 1
    writer(f"Acquired data for city: {city}")      # ← send update 2

    return f"It's always sunny in {city}!"         # ← final result

print("✅ Tool defined!")

In [ ]:
# Cell 5 — Create Agent
agent = create_agent(
    llm,
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant."
)

print("✅ Agent created!")

In [ ]:
# Cell 6 — Run with invoke() — No Streaming
# With invoke() you only see the FINAL answer
# No intermediate updates visible

print("▶️  Using invoke() — no streaming\n")

result = agent.invoke({
    "messages": [HumanMessage(content="What is the weather in Mumbai?")]
})

print("🤖 Final Answer:", result["messages"][-1].content)
print("\n⚠️  Notice: No intermediate updates were shown!")

In [ ]:
# Cell 7 — Run with stream() — See Live Updates
# With stream() + stream_mode="custom"
# you see writer() updates IN REAL TIME as tool runs

print("▶️  Using stream() — with real-time updates\n")

for chunk in agent.stream(
    {"messages": [HumanMessage(content="What is the weather in Mumbai?")]},
    stream_mode="custom",                  # ← captures writer() calls
):
    print("📡 Live update:", chunk)

print("\n✅ Done!")

# Expected output:
# ▶️  Using stream() — with real-time updates
#
# 📡 Live update: Looking up data for city: Mumbai
# 📡 Live update: Acquired data for city: Mumbai
#
# ✅ Done!

In [ ]:
# Cell 8 — Realistic Tool with Multiple Steps
# A more realistic example — simulating a long-running report tool
# with multiple progress updates

import time

@tool
def generate_sales_report(
    month   : str,
    runtime : ToolRuntime,
) -> str:
    """Generate a sales report for a given month."""
    writer = runtime.stream_writer

    writer(f"📂 Starting report generation for: {month}")
    time.sleep(0.5)                                        # simulate work

    writer(f"🔍 Fetching sales data...")
    time.sleep(0.5)

    writer(f"📊 Calculating totals and margins...")
    time.sleep(0.5)

    writer(f"📝 Formatting report...")
    time.sleep(0.5)

    writer(f"✅ Report ready!")

    return f"Sales Report — {month}: Total Revenue $1.2M, Growth +8%, Top Product: Widget Pro"


report_agent = create_agent(
    llm,
    tools=[generate_sales_report],
    system_prompt="You are a business analyst assistant."
)

print("✅ Report agent created!")

In [ ]:
# Cell 9 — Run Realistic Tool with Streaming
print("▶️  Generating Sales Report with live updates:\n")

for chunk in report_agent.stream(
    {"messages": [HumanMessage(content="Generate the sales report for March 2025")]},
    stream_mode="custom",
):
    print("📡", chunk)

print("\n✅ Streaming complete!")

# Expected output:
# ▶️  Generating Sales Report with live updates:
#
# 📡 📂 Starting report generation for: March 2025
# 📡 🔍 Fetching sales data...
# 📡 📊 Calculating totals and margins...
# 📡 📝 Formatting report...
# 📡 ✅ Report ready!
#
# ✅ Streaming complete!

In [ ]:
# Cell 10 — Capture Both Updates AND Final Answer
# stream_mode="custom" only shows writer() updates
# To get BOTH updates AND the final answer, use stream_mode=["custom","updates"]

print("▶️  Capturing both live updates AND final answer:\n")

for chunk in report_agent.stream(
    {"messages": [HumanMessage(content="Generate the sales report for April 2025")]},
    stream_mode=["custom", "updates"],     # ← capture both
):
    mode, data = chunk

    if mode == "custom":
        print(f"📡 Live  : {data}")
    elif mode == "updates":
        # final messages are in the updates
        msgs = data.get("agent", {}).get("messages", [])
        for msg in msgs:
            if hasattr(msg, "content") and msg.content:
                print(f"🤖 Answer: {msg.content}")

print("\n✅ Done!")

# Expected output:
# 📡 Live  : 📂 Starting report generation for: April 2025
# 📡 Live  : 🔍 Fetching sales data...
# 📡 Live  : 📊 Calculating totals and margins...
# 📡 Live  : 📝 Formatting report...
# 📡 Live  : ✅ Report ready!
# 🤖 Answer: Here is the sales report for April 2025: Total Revenue $1.2M...
#
# ✅ Done!

## ✅ Summary — Stream Writer

### Setup inside a tool
```python
writer = runtime.stream_writer    # get writer
writer("your message here")       # send update anytime
return "final result"             # return at the end
```

### Run the agent
```python
# Only live updates (no final answer)
agent.stream({"messages": [...]}, stream_mode="custom")

# Both live updates AND final answer
agent.stream({"messages": [...]}, stream_mode=["custom", "updates"])
```

### Key Rules
- ✅ `writer()` can be called as many times as you want
- ✅ Each `writer()` call sends one chunk to the stream
- ✅ Must use `agent.stream()` — `agent.invoke()` won't show updates
- ✅ Tool must run inside LangGraph context (`create_agent` handles this)